In [ ]:
import asyncio
import json
import os
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    executor,
    tool,
)
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")

## चरण 1: संरचित आउटपुट के लिए Pydantic मॉडल परिभाषित करें

ये मॉडल एजेंट्स द्वारा लौटाए जाने वाले **स्कीमा** को परिभाषित करते हैं। Pydantic के साथ `response_format` का उपयोग सुनिश्चित करता है:
- ✅ टाइप-सुरक्षित डेटा निष्कर्षण
- ✅ स्वचालित प्रमाणीकरण
- ✅ स्वतंत्र-पाठ प्रतिक्रियाओं से कोई पार्सिंग त्रुटियाँ नहीं
- ✅ फ़ील्ड्स के आधार पर आसान शर्तीय रूटिंग


In [ ]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

## Step 2: होटल बुकिंग टूल बनाएं

यह टूल वही है जिसे **availability_agent** कमरे उपलब्ध हैं या नहीं जांचने के लिए कॉल करेगा। हम `@ai_function` डेकोरेटर का उपयोग करते हैं ताकि:
- एक Python फ़ंक्शन को AI-कॉल करने योग्य टूल में बदला जा सके
- LLM के लिए स्वचालित रूप से JSON स्कीमा उत्पन्न किया जा सके
- पैरामीटर सत्यापन संभाला जा सके
- एजेंटों द्वारा स्वचालित रूप से कॉल करना सक्षम किया जा सके

इस डेमो के लिए:
- **स्टॉकहोम, सिएटल, टोक्यो, लंदन, एम्स्टर्डम** → कमरे उपलब्ध हैं ✅
- **सभी अन्य शहर** → कमरे उपलब्ध नहीं हैं ❌


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.

    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

## चरण 3: रूटिंग के लिए कंडीशन फंक्शंस परिभाषित करें

ये फंक्शंस एजेंट के उत्तर की जांच करते हैं और वर्कफ़्लो में कौन सा मार्ग अपनाना है तय करते हैं।

**मुख्य पैटर्न:**
1. जांचें कि संदेश `AgentExecutorResponse` है या नहीं
2. संरचित आउटपुट (Pydantic मॉडल) को पार्स करें
3. रूटिंग को नियंत्रित करने के लिए `True` या `False` लौटाएं

वर्कफ़्लो इन शर्तों का मूल्यांकन **एजों** पर करेगा ताकि यह तय किया जा सके कि अगला कौन सा एग्ज़ीक्यूटर बुलाना है।


In [ ]:
def has_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels ARE available.
    
    Returns True if the destination has hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return True  # Default to True if unexpected type

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels are NOT available.
    
    Returns True if the destination has no hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception as e:
        return False


print("✅ Condition functions defined:")
print("   - has_availability_condition (routes when rooms exist)")
print("   - no_availability_condition (routes when no rooms)")

## Step 4: कस्टम डिस्प्ले एक्सेक्यूटर बनाएं

एक्सेक्यूटर्स वेर्कफ़्लो घटक होते हैं जो ट्रांसफ़ॉर्मेशन या साइड इफेक्ट्स करते हैं। हम अंतिम परिणाम दिखाने के लिए कस्टम एक्सेक्यूटर बनाने के लिए `@executor` डेकोरेटर का उपयोग करते हैं।

**मुख्य अवधारणाएँ:**
- `@executor(id="...")` - एक फ़ंक्शन को वर्कफ़्लो एक्सेक्यूटर के रूप में पंजीकृत करता है
- `WorkflowContext[Never, str]` - इनपुट/आउटपुट के लिए टाइप हिंट्स
- `ctx.yield_output(...)` - अंतिम वर्कफ़्लो परिणाम लौटाता है


In [ ]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """
    Display the final result as workflow output.
    
    This executor receives the final agent response and yields it as the workflow output.
    """
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ display_result executor created with @executor decorator")

## Step 5: पर्यावरण चर लोड करें

LLM क्लाइंट को कॉन्फ़िगर करें। यह उदाहरण निम्नलिखित के साथ कार्य करता है:
- **GitHub मॉडल** (GitHub टोकन के साथ मुफ्त स्तर)
- **Azure OpenAI**
- **OpenAI**


In [ ]:
# Load environment variables
load_dotenv()

# Configure the Azure AI Foundry provider with keyless authentication
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## चरण 6: संरचित आउटपुट के साथ AI एजेंट बनाएँ

हम **तीन विशेषज्ञ एजेंट** बनाते हैं, जिनमें से प्रत्येक को `AgentExecutor` में लपेटा गया है:

1. **availability_agent** - टूल का उपयोग करके होटल उपलब्धता जांचता है
2. **alternative_agent** - वैकल्पिक शहर सुझाता है (जब कमरे न हों)
3. **booking_agent** - बुकिंग के लिए प्रोत्साहित करता है (जब कमरे उपलब्ध हों)

**मुख्य विशेषताएँ:**
- `tools=[hotel_booking]` - एजेंट को टूल प्रदान करता है
- `response_format=PydanticModel` - संरचित JSON आउटपुट बाध्य करता है
- `AgentExecutor(..., id="...")` - वर्कफ़्लो उपयोग के लिए एजेंट को लपेटता है


In [ ]:
# Agent 1: Check availability with tool
availability_agent = AgentExecutor(
    await provider.create_agent(
        name="availability-agent",
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), and message (string). "
            "The message should summarize the availability status."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    await provider.create_agent(
        name="alternative-agent",
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    await provider.create_agent(
        name="booking-agent",
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - Checks availability with hotel_booking tool</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)

## Step 7: कंडीशनल एजेस के साथ वर्कफ़्लो बनाएं

अब हम ग्राफ़ को कंडीशनल रूटिंग के साथ बनाने के लिए `WorkflowBuilder` का उपयोग करेंगे:

**वर्कफ़्लो संरचना:**
```
availability_agent (START)
        ↓
   Evaluate conditions
        ↙         ↘
[no_availability]  [has_availability]
        ↓              ↓
alternative_agent  booking_agent
        ↓              ↓
    display_result ←───┘
```

**मुख्य तरीके:**
- `.set_start_executor(...)` - प्रवेश बिंदु सेट करता है
- `.add_edge(from, to, condition=...)` - कंडीशनल एज जोड़ता है
- `.build()` - वर्कफ़्लो को अंतिम रूप देता है


In [ ]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing:</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> → booking_agent → display_result
        </p>
    </div>
""")
)

## Step 8: टेस्ट केस 1 चलाएँ - बिना उपलब्धता वाला शहर (पेरिस)

आइए **कोई उपलब्धता नहीं** रास्ते का परीक्षण करें पेरिस में होटल रिक्वेस्ट करके (जहाँ हमारे सिमुलेशन में कोई कमरे नहीं हैं)।


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Paris (No Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → alternative_agent → display_result</p>
    </div>
""")
)

# Create request for Paris
request_paris = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run the workflow
events_paris = await workflow.run(request_paris)
outputs_paris = events_paris.get_outputs()

# Display results
if outputs_paris:
    result_paris = AlternativeResult.model_validate_json(outputs_paris[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; box-shadow: 0 4px 12px rgba(255,165,0,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 WORKFLOW RESULT (Paris)</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Alternative Suggestion:</strong> 🏨 {result_paris.alternative_destination}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Reason:</strong> {result_paris.reason}</p>
            </div>
        </div>
    """)
    )

## Step 9: टेस्ट केस 2 चलाएँ - शहर उपलब्धता के साथ (स्टॉकहोम)

अब हम **उपलब्धता** पथ का परीक्षण करते हैं स्टॉकहोम में होटलों का अनुरोध करके (जिसमें हमारे सिमुलेशन में कमरे हैं)।


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 2: Stockholm (Has Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → booking_agent → display_result</p>
    </div>
""")
)

# Create request for Stockholm
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Stockholm")], should_respond=True
)

# Run the workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT (Stockholm)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
            </div>
        </div>
    """)
    )

## प्रमुख निष्कर्ष और अगले कदम

### ✅ आपने क्या सीखा:

1. **WorkflowBuilder पैटर्न**
   - प्रविष्टि बिंदु परिभाषित करने के लिए `.set_start_executor()` का उपयोग करें
   - कंडीशनल रूटिंग के लिए `.add_edge(from, to, condition=...)` का उपयोग करें
   - वर्कफ़्लो को अंतिम रूप देने के लिए `.build()` कॉल करें

2. **सशर्त रूटिंग**
   - कंडीशन फ़ंक्शन `AgentExecutorResponse` का निरीक्षण करते हैं
   - रूटिंग निर्णय लेने के लिए संरचित आउटपुट को पार्स करें
   - एक edge को सक्रिय करने के लिए `True` लौटाएं, उसे छोड़ने के लिए `False`

3. **टूल्स एकीकरण**
   - Python फ़ंक्शन्स को AI टूल्स में बदलने के लिए `@ai_function` का उपयोग करें
   - एजेंट आवश्यकतानुसार टूल्स को स्वचालित रूप से कॉल करते हैं
   - टूल JSON लौटाते हैं जिसे एजेंट पार्स कर सकते हैं

4. **संरचित आउटपुट**
   - टाइप-सुरक्षित डेटा निष्कर्षण के लिए Pydantic मॉडल का उपयोग करें
   - एजेंट बनाते समय `response_format=MyModel` सेट करें
   - प्रतिक्रियाओं को `Model.model_validate_json()` के साथ पार्स करें

5. **कस्टम एक्जीक्यूटर**
   - वर्कफ़्लो घटक बनाने के लिए `@executor(id="...")` का उपयोग करें
   - एक्जीक्यूटर डेटा बदल सकते हैं या साइड इफेक्ट्स कर सकते हैं
   - वर्कफ़्लो परिणाम उत्पन्न करने के लिए `ctx.yield_output()` का उपयोग करें

### 🚀 वास्तविक दुनिया के अनुप्रयोग:

- **यात्रा बुकिंग**: उपलब्धता जांचें, विकल्प सुझाएं, विकल्पों की तुलना करें
- **ग्राहक सेवा**: समस्या के प्रकार, भावना, प्राथमिकता के आधार पर मार्ग निर्धारण
- **ई-कॉमर्स**: इन्वेंटरी जांचें, विकल्प सुझाएं, ऑर्डर प्रोसेस करें
- **सामग्री मॉडरेशन**: विषाक्तता स्कोर, उपयोगकर्ता फ़्लैग के आधार पर रूटिंग
- **अनुमोदन वर्कफ़्लो**: राशि, उपयोगकर्ता भूमिका, जोखिम स्तर के आधार पर रूटिंग
- **मल्टी-स्टेज प्रोसेसिंग**: डेटा गुणवत्ता, पूर्णता के आधार पर रूटिंग

### 📚 अगले कदम:

- अधिक जटिल शर्तें जोड़ें (मल्टीपल क्राइटेरिया)
- वर्कफ़्लो स्टेट प्रबंधन के साथ लूप्स लागू करें
- पुन: उपयोग करने योग्य घटकों के लिए सब-वर्कफ़्लो जोड़ें
- वास्तविक APIs (होटल बुकिंग, इन्वेंटरी सिस्टम) के साथ एकीकरण करें
- त्रुटि हैंडलिंग और फॉलबैक पाथ जोड़ें
- अंतर्निर्मित विज़ुअलाइज़ेशन टूल्स के साथ वर्कफ़्लो का विज़ुअलाइज़ेशन करें


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
इस दस्तावेज़ का अनुवाद AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) का उपयोग करके किया गया है। जबकि हम सटीकता के लिए प्रयास करते हैं, कृपया ध्यान दें कि स्वचालित अनुवादों में त्रुटियाँ या अशुद्धियाँ हो सकती हैं। मूल दस्तावेज़ अपनी मूल भाषा में ही प्रामाणिक स्रोत माना जाना चाहिए। महत्वपूर्ण जानकारी के लिए, पेशेवर मानव अनुवाद की सिफारिश की जाती है। इस अनुवाद के उपयोग से उत्पन्न किसी भी गलतफहमी या गलत व्याख्या के लिए हम उत्तरदायी नहीं हैं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
